# Logits Preprocessing and Data Engineering

In [ ]:
def default_params(): 
    return {
        'current_model': 'M1', 
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/data/extension/mitigation/datasets',
            'current': 'prompted', # 'base' or 'prompted',
            'content_column': 'code',
            'sampling_size': 500,
            'prompt_column': 'prompt',
        },
        'logging_path': '/workspaces/CodeSmells/datax/code_smells/logs', 
        'output_path' : '/workspaces/CodeSmells/datax/code_smells/logits/mitigation',
        'callbacks_path' : '/workspaces/CodeSmells/datax/code_smells/callbacks/mitigation',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
        'causal_models': {
            ##### BY ARCHITECTURE, SAME SIZE #####
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
            'M2' : 'mistralai/Mistral-7B-v0.3', #https://huggingface.co/mistralai/Mistral-7B-v0.3,
            'M3' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B,
            'M4' : 'bigcode/starcoder2-7b', #https://huggingface.co/bigcode/starcoder2-7b,
            ##### BY SIZE, SAME ARCHITECTURE #####
            'S1' : 'Qwen/Qwen2.5-Coder-0.5B', #https://huggingface.co/Qwen/Qwen2.5-Coder-0.5B,
            'S2' : 'Qwen/Qwen2.5-Coder-1.5B', #https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B,
            'S3' : 'Qwen/Qwen2.5-Coder-3B', #https://huggingface.co/Qwen/Qwen2.5-Coder-3B,
            'S4' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B,
        },
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import os
import time
import numpy as np
import torch
import gc

In [3]:
import seaborn as sns
from scipy import stats
from statistics import NormalDist
import matplotlib.pyplot as plt

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

In [5]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [6]:
# Define log file path
log_file = f"{params['logging_path']}/{params['current_model']}"
create_folder(log_file)
log_file += '/data_en.txt'

# Create the log file if it doesn't exist
if not os.path.exists(log_file):
    with open(log_file, 'w'): 
        pass  # Create an empty log file

In [7]:
import logging
logging.basicConfig(filename=log_file, format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

#### Dataset

In [8]:
print(f"{params['dataset']['path']}/{params['dataset']['current']}.json")

/workspaces/CodeSmells/data/extension/mitigation/datasets/base.json


In [9]:
df_dataset = pd.read_json(f"{params['dataset']['path']}/{params['dataset']['current']}.json", )

In [10]:
df_dataset

,id,commit_id,repo,path,file_name,fun_name,commit_message,code,url,language,...,n_ast_nodes,n_identifiers,s_msg_id,s_line,s_column,s_end_line,s_end_column,s_code,category,input_lenght
0,278224,ebb5e0e3f5e0b02cad2b54144022084301588ac5,keras,keras/mixed_precision/loss_scale_optimizer_tes...,loss_scale_optimizer_test.py,testHyperParametersExposed,resolve line-too-long in mixed_precision,def testHyperParametersExposed(self):\n ...,https://github.com/keras-team/keras.git,Python,...,456,22,C0304,30,0,30,73,# LossScaleOptimizer and hyperpara...,Convention,536
1,276644,84afc5193d38057e2e2badf9c889ea87d80d8fbf,keras,keras/tests/tracking_util_with_v1_optimizers_t...,tracking_util_with_v1_optimizers_test.py,__init__,Reformatting the codebase with black.\n\nPiper...,def __init__(self):\n super().__init__(...,https://github.com/keras-team/keras.git,Python,...,53,8,C0304,7,0,7,30,# pylint: disable=not-callable,Convention,62
2,24968,1a0a75e3fa896cf3c095e4146a54acef7223657e,PaddleOCR,ppstructure/pdf2word/pdf2word.py,pdf2word.py,predictAndSave,Add pdf2word exe\n\nAdd pdf2word exe,"def predictAndSave(self, imgs, img_name):\r\n ...",https://github.com/PaddlePaddle/PaddleOCR.git,Python,...,251,34,C0304,25,0,25,0,,Convention,308
3,278875,3613c3defc39c236fb1592c4f7ba1a9cc887343a,keras,keras/mixed_precision/loss_scale_optimizer_tes...,loss_scale_optimizer_test.py,testHyperParametersExposed,Remove pylint comments.\n\nPiperOrigin-RevId: ...,def testHyperParametersExposed(self):\n ...,https://github.com/keras-team/keras.git,Python,...,455,22,C0304,30,0,30,73,# LossScaleOptimizer and hyperpara...,Convention,524
4,152057,a6adc22f0711c8ab78c6ef8fc78715f815cc750f,stable-diffusion-webui,webui.py,webui.py,js,added interrupt button\nadded save button\n--a...,def js(self):\r\n obj = {\r\n ...,https://github.com/AUTOMATIC1111/stable-diffus...,Python,...,102,13,C0304,11,0,11,30,return json.dumps(obj),Convention,101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,115229,4f2861b6ded274d4a41322c107ace8107e86ebea,mindsdb,mindsdb/interfaces/database/views.py,views.py,add,store integration in sql of view (before save it),"def add(self, name, query, integration_name, c...",https://github.com/mindsdb/mindsdb.git,Python,...,247,25,W0719,21,16,21,88,"raise Exception(f""Can't find integration with ...",Warning,264
1496,282637,fae93c67adc9015c1466712f9c8ffa35a8b70872,OpenBBTerminal,bots/economy/usbonds.py,usbonds.py,usbonds_command,Refactor Bot (#1326)\n\n* First commit\r\n\r\n...,def usbonds_command():\n \n\n # Debug us...,https://github.com/OpenBB-finance/OpenBBTermin...,Python,...,469,45,W0719,12,8,12,50,"raise Exception(""No available data found"")",Warning,545
1497,2842,ccd4b9330a186090cc87e94d2da1093d45de329f,PySyft,packages/syft/src/syft/oblv/model.py,model.py,request_publish,Changes for model,"def request_publish(self, dataset_id, sigma = ...",https://github.com/OpenMined/PySyft.git,Python,...,383,31,W0719,2,12,2,116,"raise Exception(""No Domain Clients added. Set ...",Warning,497
1498,116308,326622a6fb33664de21ee1627f5f083f11b59a9e,mindsdb,mindsdb/integrations/handlers/ludwig_handler/l...,ludwig_handler.py,_learn,fix: add hyperopt,"def _learn(self, statement):\n model_na...",https://github.com/mindsdb/mindsdb.git,Python,...,374,50,W0719,8,12,8,85,"raise Exception(""Ludwig handler does not suppo...",Warning,466


#### Model Loading

In [11]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_dir, use_fast=True)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     if params['quantization'] == 'int4':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
     elif params['quantization'] == 'int8':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
     elif params['quantization'] == 'float32':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
     elif params['quantization'] == 'float16':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
     else: 
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [12]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [13]:
print(model.config)
print(model.__class__)
print(tokenizer.__class__)

LlamaConfig {
  "_name_or_path": "codellama/CodeLlama-7b-hf",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 16384,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 1000000,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.36.2",
  "use_cache": true,
  "vocab_size": 32016
}

<class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>
<class 'transformers.models.code_llama.tokenization_code_llama_fast.CodeLlamaTokenizerFast'>


#### preprocess dataset

In [14]:
df_dataset['input_ids'] = df_dataset[params['dataset']['content_column']].map(lambda code: tokenizer.encode(code, add_special_tokens=False))
if params['dataset']['current'] == 'prompted':
    df_dataset['prompt_ids'] = df_dataset[params['dataset']['prompt_column']].map(lambda code: tokenizer.encode(code, add_special_tokens=False))
df_dataset['input_lenght'] = df_dataset['input_ids'].map(lambda input_ids: len(input_ids))

#### Softmax Normalization and Data Engineering

In [15]:
def topk_tuple(logit_vocab_tensor, largest, tokenizer_fn):
    """
    Return the decoded top-1 (or bottom-1) token and its logit value.
    """
    topk = logit_vocab_tensor.topk(k=1, largest=largest)
    top_token_id = topk.indices[0].item()
    decoded_token = tokenizer_fn.decode([top_token_id])  # Already handles special tokens and spaces
    return (decoded_token, topk.values[0].item())

In [16]:
def analyze_logits(logit_tensor_sequence, input_token_ids, tokenizer_fn, skip_first_token=True):
    """
    Analyze logits for each token in a sequence.

    Args:
        logit_tensor_sequence (List[Tensor]): List of vocab-sized logits for each token position.
        input_token_ids (List[int] or Tensor): Token IDs of the input prompt.
        tokenizer_fn: HuggingFace tokenizer with .decode() method.
        skip_first_token (bool): Whether to skip the first token prediction (default: True).

    Returns:
        dict with:
            - "max_cases": list of (decoded top-1 token, logit value)
            - "min_cases": list of (decoded bottom-1 token, logit value)
            - "actual_logits": list of (decoded ground-truth token, logit value)
    """
    max_cases = []
    min_cases = []
    actual_logits = []

    start_index = 1 if skip_first_token else 0
    token_targets = input_token_ids[start_index:]

    for position, token_id in enumerate(token_targets):
        vocab_logits = logit_tensor_sequence[position]

        # Top-1 max and min predictions
        max_case = topk_tuple(logit_vocab_tensor=vocab_logits, largest=True, tokenizer_fn=tokenizer_fn)
        min_case = topk_tuple(logit_vocab_tensor=vocab_logits, largest=False, tokenizer_fn=tokenizer_fn)

        # Actual token logit
        decoded_token = tokenizer_fn.decode([int(token_id)])
        logit_value = vocab_logits[int(token_id)].item()
        actual_case = (decoded_token, logit_value)

        max_cases.append(max_case)
        min_cases.append(min_case)
        actual_logits.append(actual_case)

    return {
        "max_cases": max_cases,
        "min_cases": min_cases,
        "actual_logits": actual_logits
    }

In [17]:
soft = torch.nn.Softmax( dim = 0 ) #Flattening normalization

In [18]:
callbacks_dir = f"{params['callbacks_path']}/{params['dataset']['current']}/{params['current_model']}_q_{params['quantization']}"
out = np.load(f"{callbacks_dir}/logits_tensor[0]_batch[0].npy")

print(out.shape) #<sample,tokens,voc_tokens>
out = out[0]


(1, 536, 32016)


In [19]:
input_ids_list = tokenizer.batch_encode_plus(df_dataset[params['dataset']['content_column']].tolist())
input_ids_list = [torch.tensor(  input_ids, dtype = torch.int) for input_ids in input_ids_list.input_ids]

logit_dict = analyze_logits(
    logit_tensor_sequence = [ soft( torch.from_numpy(token) ) for token in out] , #Out is a complete sequence
    input_token_ids = input_ids_list[0], ## SAMPLE ID
    tokenizer_fn = tokenizer
)

2025-11-18 19:42:34.661716: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.11/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/usr

In [20]:
assert len(set(len(v) for v in logit_dict.values())) == 1, "All key array values in logit_dict do not have the same length"

#### Processing all the Batches

In [21]:
def process_logit_batches(tokenizer, tokenized_inputs, num_samples=10000, skip_first_token=True):
    """
    Process multiple saved logits files and extract:
    - top-1 max logit predictions,
    - top-1 min logit predictions,
    - actual logits for ground-truth tokens.

    Args:
        tokenizer: HuggingFace tokenizer instance.
        tokenized_inputs (List[Tensor]): Tokenized input prompts (one per sample).
        num_samples (int): Number of samples to process.
        skip_first_token (bool): Whether to skip the first token prediction.

    Returns:
        Tuple of lists: (max_logit_predictions, min_logit_predictions, actual_logits)
    """
    max_logit_predictions = []
    min_logit_predictions = []
    actual_logit_scores = []

    softmax_fn = torch.nn.Softmax(dim=0)
    base_path = f"{params['callbacks_path']}/{params['dataset']['current']}/{params['current_model']}_q_{params['quantization']}"

    for sample_idx in range(num_samples):
        logits_file_path = f"{base_path}/logits_tensor[{sample_idx}]_batch[{sample_idx}].npy"
        logits_array = np.load(logits_file_path)[0]  # Shape: [sequence_length, vocab_size]

        # Apply softmax to each token’s logits
        normalized_logits = [softmax_fn(torch.from_numpy(token_logits)) for token_logits in logits_array]

        # Analyze logits using the unified function
        result = analyze_logits(
            logit_tensor_sequence=normalized_logits,
            input_token_ids=tokenized_inputs[sample_idx],
            tokenizer_fn=tokenizer,
            skip_first_token=skip_first_token
        )

        max_logit_predictions.append(result["max_cases"])
        min_logit_predictions.append(result["min_cases"])
        actual_logit_scores.append(result["actual_logits"])

        logging.info(f"Processed sample {sample_idx}")
        print(f"Processed sample {sample_idx}")

    return max_logit_predictions, min_logit_predictions, actual_logit_scores

In [22]:
input_ids_list = tokenizer.batch_encode_plus(df_dataset[params['dataset']['content_column']].tolist())
input_ids_list = [torch.tensor(  input_ids, dtype = torch.int) for input_ids in input_ids_list.input_ids]

In [23]:
max_logit_token_prompt, min_logit_token_prompt, actual_logit_token_prompt = process_logit_batches(
    tokenizer=tokenizer , tokenized_inputs=input_ids_list, 
    num_samples = len(df_dataset)
) #<---WARNING TIME Consuming

Processed sample 0
Processed sample 1
Processed sample 2
Processed sample 3
Processed sample 4
Processed sample 5
Processed sample 6
Processed sample 7
Processed sample 8
Processed sample 9
Processed sample 10
Processed sample 11
Processed sample 12
Processed sample 13
Processed sample 14
Processed sample 15
Processed sample 16
Processed sample 17
Processed sample 18
Processed sample 19
Processed sample 20
Processed sample 21
Processed sample 22
Processed sample 23
Processed sample 24
Processed sample 25
Processed sample 26
Processed sample 27
Processed sample 28
Processed sample 29
Processed sample 30
Processed sample 31
Processed sample 32
Processed sample 33
Processed sample 34
Processed sample 35
Processed sample 36
Processed sample 37
Processed sample 38
Processed sample 39
Processed sample 40
Processed sample 41
Processed sample 42
Processed sample 43
Processed sample 44
Processed sample 45
Processed sample 46
Processed sample 47
Processed sample 48
Processed sample 49
Processed 

#### Saving results

In [24]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [25]:
dataframe_to_save = df_dataset.copy()
dataframe_to_save['max_prob'] = max_logit_token_prompt
dataframe_to_save['min_prob'] = min_logit_token_prompt
dataframe_to_save['actual_prob'] = actual_logit_token_prompt
dataframe_to_save.shape

(1500, 33)

In [26]:
output_dir = f"{params['output_path']}/{params['dataset']['current']}/{params['current_model']}_q_{params['quantization']}"
create_folder(output_dir)
dataframe_to_save.to_json(f"{output_dir}/raw_logits.json", index=False)

#### Loss Retrieval

In [27]:
def batching_loss( size = dataframe_to_save.shape[0] ):
    output_dir = f"{params['callbacks_path']}/{params['dataset']['current']}/{params['current_model']}_q_{params['quantization']}"
    output_loss = []
    for current_batch in range(size):
        out = np.load(f"{output_dir}/_loss_batch[{current_batch}].npy")
        output_loss.append( out.item() ) #.item() for numpy library
        logging.info(current_batch)
    return output_loss

In [28]:
output_loss = batching_loss() #[WAENING!] Takes Time

In [29]:
output_loss

[0.4817667007446289,
 1.2113902568817139,
 1.3814879655838013,
 0.48720982670783997,
 1.1836971044540405,
 0.9220165610313416,
 1.0162023305892944,
 0.9585427045822144,
 2.3622360229492188,
 1.4342334270477295,
 1.7954044342041016,
 1.1472768783569336,
 2.301631212234497,
 3.1858749389648438,
 0.9545468091964722,
 1.8392764329910278,
 1.280494213104248,
 1.1617343425750732,
 0.2592431902885437,
 0.6597101092338562,
 1.9887151718139648,
 1.833314061164856,
 1.473940372467041,
 1.7954044342041016,
 0.5083361864089966,
 2.794299364089966,
 1.626961350440979,
 0.9804427623748779,
 1.434414267539978,
 0.30446356534957886,
 0.6057441234588623,
 1.3440114259719849,
 1.1693793535232544,
 1.122738003730774,
 0.8383280634880066,
 0.878592848777771,
 0.6235223412513733,
 1.8597307205200195,
 1.84883451461792,
 0.9256730079650879,
 1.0020115375518799,
 0.7943770289421082,
 0.7912768721580505,
 1.579355239868164,
 1.360980749130249,
 0.9012847542762756,
 1.9246673583984375,
 2.067991018295288,
 0.9

In [30]:
dataframe_to_save['loss'] = output_loss
dataframe_to_save.head(5)

,id,commit_id,repo,path,file_name,fun_name,commit_message,code,url,language,...,s_end_line,s_end_column,s_code,category,input_lenght,input_ids,max_prob,min_prob,actual_prob,loss
0,278224,ebb5e0e3f5e0b02cad2b54144022084301588ac5,keras,keras/mixed_precision/loss_scale_optimizer_tes...,loss_scale_optimizer_test.py,testHyperParametersExposed,resolve line-too-long in mixed_precision,def testHyperParametersExposed(self):\n ...,https://github.com/keras-team/keras.git,Python,...,30,73,# LossScaleOptimizer and hyperpara...,Convention,535,"[822, 1243, 26322, 546, 11507, 1252, 4752, 298...","[(<PRE>, 0.7492900490760803), (module, 0.47944...","[(<s>, 6.321457571810407e-13), ($}, 1.21011436...","[(def, 0.0007611316395923495), (test, 0.019638...",0.481767
1,276644,84afc5193d38057e2e2badf9c889ea87d80d8fbf,keras,keras/tests/tracking_util_with_v1_optimizers_t...,tracking_util_with_v1_optimizers_test.py,__init__,Reformatting the codebase with black.\n\nPiper...,def __init__(self):\n super().__init__(...,https://github.com/keras-team/keras.git,Python,...,7,30,# pylint: disable=not-callable,Convention,61,"[822, 4770, 2344, 12035, 1311, 1125, 13, 4706,...","[(<PRE>, 0.7492942214012146), (module, 0.47944...","[(<s>, 6.322590020979568e-13), ($}, 1.21010909...","[(def, 0.0007611323380842805), (__, 0.00598202...",1.211390
2,24968,1a0a75e3fa896cf3c095e4146a54acef7223657e,PaddleOCR,ppstructure/pdf2word/pdf2word.py,pdf2word.py,predictAndSave,Add pdf2word exe\n\nAdd pdf2word exe,"def predictAndSave(self, imgs, img_name):\r\n ...",https://github.com/PaddlePaddle/PaddleOCR.git,Python,...,25,0,,Convention,307,"[822, 8500, 2855, 11371, 29898, 1311, 29892, 5...","[(<PRE>, 0.7492886185646057), (module, 0.47944...","[(<s>, 6.321602312800434e-13), ($}, 1.21012630...","[(def, 0.0007611338514834642), (predict, 0.000...",1.381488
3,278875,3613c3defc39c236fb1592c4f7ba1a9cc887343a,keras,keras/mixed_precision/loss_scale_optimizer_tes...,loss_scale_optimizer_test.py,testHyperParametersExposed,Remove pylint comments.\n\nPiperOrigin-RevId: ...,def testHyperParametersExposed(self):\n ...,https://github.com/keras-team/keras.git,Python,...,30,73,# LossScaleOptimizer and hyperpara...,Convention,523,"[822, 1243, 26322, 546, 11507, 1252, 4752, 298...","[(<PRE>, 0.7492900490760803), (module, 0.47944...","[(<s>, 6.321457571810407e-13), ($}, 1.21011436...","[(def, 0.0007611316395923495), (test, 0.019638...",0.487210
4,152057,a6adc22f0711c8ab78c6ef8fc78715f815cc750f,stable-diffusion-webui,webui.py,webui.py,js,added interrupt button\nadded save button\n--a...,def js(self):\r\n obj = {\r\n ...,https://github.com/AUTOMATIC1111/stable-diffus...,Python,...,11,30,return json.dumps(obj),Convention,100,"[822, 6965, 29898, 1311, 1125, 30004, 13, 4706...","[(<PRE>, 0.749289870262146), (module, 0.479445...","[(<s>, 6.321889626376143e-13), ($}, 1.21010756...","[(def, 0.0007611394976265728), (js, 1.99369169...",1.183697


In [31]:
## Saving CheckPoint 2
dataframe_to_save.to_json(f"{output_dir}/raw_logits.json", index=False)

In [32]:
print("================================= PROCESS COMPLETED =================================")

================================= PROCESS COMPLETED =================================


In [33]:
del model
torch.cuda.empty_cache()
gc.collect()

352